# Query performance / timeout tuning

Uses `execute_query`'s own `elapsed_ms` (not a separate stopwatch) to benchmark realistic agent-generated queries, compares an indexed vs. unindexed filter, inspects query plans, and probes where `query_timeout_s` needs to sit so real joins pass but runaway cartesian products don't.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from db import execute_query, fetch_all
from config import settings

settings.query_timeout_s

10.0

In [2]:
# a handful of realistic multi-table questions, timed via QueryResult.elapsed_ms
queries = {
    "revenue_by_category": """
        SELECT t.product_category_name_english, round(sum(oi.price), 2) AS revenue
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_translation t ON p.product_category_name = t.product_category_name
        GROUP BY 1 ORDER BY revenue DESC LIMIT 10
    """,
    "avg_review_by_state": """
        SELECT c.customer_state, round(avg(r.review_score), 2) AS avg_score, count(*) AS n
        FROM order_reviews r
        JOIN orders o ON r.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY 1 ORDER BY n DESC LIMIT 10
    """,
    "late_deliveries": """
        SELECT count(*) FROM orders
        WHERE order_delivered_customer_date > order_estimated_delivery_date
    """,
    "seller_geolocation_join": """
        SELECT s.seller_state, count(DISTINCT s.seller_id) AS n_sellers
        FROM sellers s
        JOIN geolocation g ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
        GROUP BY 1 ORDER BY n_sellers DESC LIMIT 10
    """,
}

results = []
for name, sql in queries.items():
    r = execute_query(sql, unlimited=True)
    results.append((name, r.ok, r.row_count, round(r.elapsed_ms, 2)))

pd.DataFrame(results, columns=["query", "ok", "row_count", "elapsed_ms"])

,query,ok,row_count,elapsed_ms
0,revenue_by_category,True,10,149.76
1,avg_review_by_state,True,10,433.84
2,late_deliveries,True,1,6.07
3,seller_geolocation_join,True,10,72.16


In [3]:
# indexed vs unindexed filter on the same table (order_items has an index on
# product_id and seller_id, but not on price)
indexed = execute_query("SELECT * FROM order_items WHERE product_id = '1e9e8ef04dbcff4541ed26657ea517e5'")
unindexed = execute_query("SELECT * FROM order_items WHERE price > 500", unlimited=True)

print(f"indexed  (product_id filter): {indexed.row_count} rows in {indexed.elapsed_ms:.2f} ms")
print(f"unindexed (price filter):     {unindexed.row_count} rows in {unindexed.elapsed_ms:.2f} ms")

indexed  (product_id filter): 1 rows in 0.22 ms
unindexed (price filter):     3216 rows in 7.43 ms


In [4]:
# EXPLAIN QUERY PLAN for both, to see whether SQLite actually used the index
print("-- indexed --")
for row in fetch_all("EXPLAIN QUERY PLAN SELECT * FROM order_items WHERE product_id = '1e9e8ef04dbcff4541ed26657ea517e5'"):
    print(row)

print("\n-- unindexed --")
for row in fetch_all("EXPLAIN QUERY PLAN SELECT * FROM order_items WHERE price > 500"):
    print(row)

-- indexed --
(3, 0, 50, 'SEARCH order_items USING INDEX idx_order_items_product_id (product_id=?)')

-- unindexed --
(2, 0, 181, 'SCAN order_items')


In [5]:
# find where the timeout needs to sit: a missing-join-predicate cartesian
# product (orders x order_items, ~11B row pairs) should be cut off quickly,
# while a real join with a similar row count should comfortably finish
cartesian = execute_query(
    "SELECT count(*) FROM orders a, order_items b", timeout_s=2.0
)
real_join = execute_query(
    "SELECT count(*) FROM orders a JOIN order_items b ON a.order_id = b.order_id",
    timeout_s=2.0,
)

print("cartesian (should time out):", cartesian.preview())
print("real join (should succeed): ", real_join.preview(), f"({real_join.elapsed_ms:.1f} ms)")

cartesian (should time out): ERROR: Query exceeded the 2s time limit and was cancelled. It is probably missing a join condition or aggregating over too many rows -- add the join predicate or narrow the filter.
real join (should succeed):  1 rows | 112650 (17.9 ms)


**Takeaway:** the real multi-table joins above all finish in low single-digit milliseconds (indexes are doing their job), while the accidental cartesian product is reliably killed well inside a 2s window. `settings.query_timeout_s = 10.0` has a lot of headroom above real query costs and mainly exists to bound pathological agent-generated SQL — no need to tighten it for these table sizes.